In [1]:
# Processing monthly PM2.5 and save as annual mean

In [2]:
import os
import json
import xarray as xr
from utils.utils import get_scenario_config, fix_months

In [ ]:
def load_file_list(DIR, filename):
    file_path = os.path.join(DIR, filename)
    with open(file_path, "r") as f:
        data = json.load(f)
    return data["files"]

In [4]:
# === Processing Function ===
def calculate_annual_surface_pm25(pm25_surf):
    # Compute annual mean
    print("Calculating annual mean")
    annual_mean = pm25_surf.groupby("time.year").mean("time")
    return annual_mean

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "hist"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_PM25/"

# === Main Loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")
    datasets = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        # Take the surface pressure value
        datasets.append(xr.open_dataset(f)["PM25"][:, -1, :, :])

    # Combine files if multiple
    combined_ds = xr.concat(datasets,
                            dim="time") if len(datasets) > 1 else datasets[0]

    if scenario == "ARISE":
        try:
            expected_end = "2070-12" if ens_num in [8, 9] else "2069-12"
            pm_surf = fix_months(combined_ds, "2035-01",
                                 expected_end, scenario)
            annual_pm25 = calculate_annual_surface_pm25(pm_surf)
        except Exception as e:
            print(f"Error processing ensemble {ens_num:02d}: {e}")
            continue

    if scenario == "SSP245":
        try:
            expected_end = "2100-12" if ens_num <= 5 else "2069-12"
            pm_surf = fix_months(combined_ds, "2015-01",
                                 expected_end, scenario)
            annual_pm25 = calculate_annual_surface_pm25(pm_surf)
        except Exception as e:
            print(f"Error processing ensemble {ens_num:02d}: {e}")
            continue

    # Save output
    if scenario == "ARISE":
        dates = "2035-2069"
    elif scenario == "SSP245":
        dates = "2020-2069"

    out_file = f"PM25_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    description = ("Annual mean PM2.5 - scripts by A.F. Wells (2025)")
    annual_pm25.attrs = combined_ds.attrs
    annual_pm25.attrs["description"] = description
    annual_pm25.attrs["ensemble_number"] = ens_num
    annual_pm25.attrs["scenario"] = scenario

    print(f"Saving annual PM2.5 to {out_path}")
    annual_pm25.to_netcdf(out_path)

print("Done processing all PM2.5 ensembles.")

Processing ARISE, ensemble 01
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.001.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_01_2035-2069.nc
Processing ARISE, ensemble 02
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.002.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_02_2035-2069.nc
Processing ARISE, ensemble 03
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.003.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_03_2035-2069.nc
Processing ARISE, ensemble 04
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.004.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_04_2035-2069.nc
Processi